# AnimalEmotion AI – Deep Learning-Based Animal Emotion Recognition System
## Phase 1: Dataset Audit

Evidence-driven dataset auditing only. No model training, data splitting, or modification/deletion of source images.

Dataset page: https://www.kaggle.com/datasets/kk2004/animalemotion/data

## Manual Kaggle ZIP workflow
1. Download the dataset ZIP manually from the Kaggle dataset page and accept any Kaggle terms there.
2. Run the next cell and select exactly one `.zip` file.
3. The notebook retains the ZIP under `/content/animalemotion/uploads/` and extracts it to `/content/animalemotion/dataset/`. Reports are written to `/content/animalemotion/reports/`.
4. `DATA_DIR` must be empty before extraction. Existing files stop the workflow; use a fresh Colab runtime or manually clear an unwanted dataset directory. Files are never deleted automatically.
5. `files.upload()` loads the ZIP into Colab memory. Configure `MAX_RECOMMENDED_ZIP_MB` below to show a warning for large uploads. This warning does not change or replace the upload workflow.

In [ ]:
from pathlib import Path
import os, sys, json, zipfile, hashlib, stat, shutil, random, re, subprocess, datetime, platform, ast
import numpy as np
import pandas as npd
import pandas as pd
from PIL import Image, ImageOps, UnidentifiedImageError, ImageFile
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import torch, torchvision, cv2, sklearn
try:
    import psutil
except Exception:
    psutil=None
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
sns.set_theme(style='whitegrid')
PROJECT_DIR=Path('/content/animalemotion')
UPLOAD_DIR=PROJECT_DIR/'uploads'
DATA_DIR=PROJECT_DIR/'dataset'
REPORT_DIR=PROJECT_DIR/'reports'
for directory in (PROJECT_DIR,UPLOAD_DIR,DATA_DIR,REPORT_DIR): directory.mkdir(parents=True,exist_ok=True)
MAX_RECOMMENDED_ZIP_MB=500
DATASET_SOURCE_URL='https://www.kaggle.com/datasets/kk2004/animalemotion/data'
DATASET_SLUG='kk2004/animalemotion'
resource_log=[]
def resource_snapshot(stage):
    row={'stage':stage,'utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'rss_bytes':None,'available_memory_bytes':None}
    if psutil:
        try:
            proc=psutil.Process(); row.update(rss_bytes=proc.memory_info().rss,available_memory_bytes=psutil.virtual_memory().available)
        except Exception as exc: row['error']=str(exc)
    return row
resource_log.append(resource_snapshot('setup'))
print('Python',platform.python_version(),'| torch',torch.__version__,'| CUDA',torch.cuda.is_available())

## Safe manual ZIP upload and extraction

In [ ]:
from google.colab import files

# DATA_DIR must be empty to prevent accidental mixing of dataset versions.
existing_data=[p for p in DATA_DIR.iterdir()]
if existing_data:
    sample=', '.join(p.name for p in existing_data[:10])
    raise RuntimeError(f'DATA_DIR is not empty ({len(existing_data)} top-level entries, such as {sample}). Stop and use a fresh Colab runtime or manually point DATA_DIR to an empty directory. Existing files were not deleted.')

print('Upload exactly one .zip downloaded manually from the Kaggle dataset page.')
uploaded=files.upload()
if not uploaded:
    raise RuntimeError('No file uploaded. Run this cell again and select exactly one ZIP.')
zip_names=[name for name in uploaded if str(name).lower().endswith('.zip')]
if len(uploaded)!=1 or len(zip_names)!=1:
    raise ValueError(f'Expected exactly one uploaded ZIP; received {len(uploaded)} file(s), {len(zip_names)} ZIP(s).')

original_name=Path(zip_names[0]).name
if not original_name:
    raise ValueError('Uploaded filename is empty or invalid.')
archive_bytes=uploaded[zip_names[0]]
if not archive_bytes:
    raise ValueError('Uploaded ZIP is empty.')
archive_mb=len(archive_bytes)/1024/1024
if archive_mb>MAX_RECOMMENDED_ZIP_MB:
    print(f'WARNING: uploaded ZIP is {archive_mb:.1f} MB, above the configurable {MAX_RECOMMENDED_ZIP_MB} MB recommendation. files.upload() loads it into Colab memory; large uploads may fail. Use a smaller/available runtime or keep the ZIP under the warning threshold.')
else:
    print(f'Uploaded ZIP size: {archive_mb:.1f} MB')

archive_digest=hashlib.sha256(archive_bytes).hexdigest()
zip_path=UPLOAD_DIR/original_name
if zip_path.exists():
    existing_digest=hashlib.sha256(zip_path.read_bytes()).hexdigest()
    if existing_digest==archive_digest:
        print('A byte-identical ZIP already exists; the existing retained file will be used.')
    else:
        stem,suffix=zip_path.stem,zip_path.suffix
        i=1
        while True:
            candidate=UPLOAD_DIR/f'{stem}_{i}{suffix}'
            if not candidate.exists():
                zip_path=candidate
                break
            i+=1
        with open(zip_path,'xb') as stream:
            stream.write(archive_bytes)
else:
    with open(zip_path,'xb') as stream:
        stream.write(archive_bytes)

print('Uploaded ZIP retained at:',zip_path)
if not zipfile.is_zipfile(zip_path):
    raise ValueError('The uploaded file has a .zip extension but is not a valid ZIP archive.')

def safe_extract(archive_path,destination):
    root=destination.resolve(); entries=[]; names=set(); rejected=[]
    with zipfile.ZipFile(archive_path,'r') as archive:
        bad=archive.testzip()
        if bad is not None:
            raise ValueError(f'ZIP CRC check failed for member: {bad}')
        for member in archive.infolist():
            name=member.filename; rel=Path(name); mode=member.external_attr>>16
            if name in names:
                raise ValueError(f'Duplicate ZIP member name: {name}')
            names.add(name)
            if name.startswith('/') or re.match(r'^[A-Za-z]:',name) or '..' in rel.parts or stat.S_ISLNK(mode):
                rejected.append({'entry':name,'reason':'unsafe path or symbolic link'}); continue
            target=(root/rel).resolve()
            if target!=root and root not in target.parents:
                rejected.append({'entry':name,'reason':'resolved outside DATA_DIR'}); continue
            if target.exists() and not member.is_dir():
                raise FileExistsError(f'Extraction would overwrite existing data: {target}. Nothing was overwritten.')
            entries.append((member,target))
        if rejected:
            raise ValueError('Unsafe ZIP entries rejected: '+json.dumps(rejected[:20]))
        written=[]
        try:
            for member,target in entries:
                if member.is_dir(): target.mkdir(parents=True,exist_ok=True); continue
                target.parent.mkdir(parents=True,exist_ok=True)
                with archive.open(member,'r') as src, open(target,'xb') as dst:
                    shutil.copyfileobj(src,dst)
                written.append(target)
        except Exception as exc:
            raise RuntimeError(f'Extraction stopped at {target}; earlier files remain. Cause: {type(exc).__name__}: {exc}') from exc
    return written

try:
    extracted_files=safe_extract(zip_path,DATA_DIR)
except zipfile.BadZipFile as exc:
    raise ValueError(f'Invalid or damaged ZIP archive: {exc}') from exc
verification_count=sum(1 for p in DATA_DIR.rglob('*') if p.is_file())
if verification_count==0:
    raise RuntimeError('ZIP extraction verification failed: DATA_DIR contains no files.')
upload_metadata={'dataset_slug':DATASET_SLUG,'source_url':DATASET_SOURCE_URL,'uploaded_filename':zip_path.name,'archive_sha256':archive_digest,'archive_bytes':zip_path.stat().st_size,'upload_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'dataset_version':'unknown; manual download version not embedded/verified','license_metadata':'manual Kaggle page review required','usage_review':{'reviewed':False,'status':'manual review required'}}
(REPORT_DIR/'dataset_metadata_and_license_review.json').write_text(json.dumps(upload_metadata,indent=2))
(REPORT_DIR/'safe_extraction.json').write_text(json.dumps({'archive':zip_path.name,'extracted_file_count':len(extracted_files),'extraction_root':str(DATA_DIR),'verification_file_count':verification_count,'rejected_entries':[]},indent=2))
resource_log.append(resource_snapshot('after_extraction'))
print('Extraction verified:',verification_count,'files | DATA_DIR:',DATA_DIR)

## Inventory, image integrity, dimensions, and duplicates

In [ ]:
IMAGE_EXTS={'.jpg','.jpeg','.png','.bmp','.gif','.tif','.tiff','.webp','.jfif','.heic','.heif'}
ANNOT_EXTS={'.csv','.json','.xlsx','.xls','.yaml','.yml','.xml','.txt'}
all_files=[p for p in DATA_DIR.rglob('*') if p.is_file()]
image_files=[p for p in all_files if p.suffix.lower() in IMAGE_EXTS]
annotation_files=[p for p in all_files if p.suffix.lower() in ANNOT_EXTS]
folder_counts={}
for path in all_files:
    rel=path.relative_to(DATA_DIR)
    for depth in range(1,len(rel.parts)):
        key='/'.join(rel.parts[:depth]); folder_counts[key]=folder_counts.get(key,0)+1
folder_df=pd.DataFrame([{'folder':k,'file_count':v} for k,v in sorted(folder_counts.items())],columns=['folder','file_count'])
if folder_df.empty: folder_df=pd.DataFrame({'folder':pd.Series(dtype='string'),'file_count':pd.Series(dtype='int64')})
folder_df.to_csv(REPORT_DIR/'folder_hierarchy.csv',index=False)
IMAGE_COLUMNS=['relative_path','filename','extension','status','format','mode','channels','width','height','aspect_ratio','file_bytes','sha256','has_transparency','error']
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
def inspect_image(path):
    row={'relative_path':path.relative_to(DATA_DIR).as_posix(),'filename':path.name,'extension':path.suffix.lower(),'status':'unreadable','format':None,'mode':None,'channels':None,'width':None,'height':None,'aspect_ratio':None,'file_bytes':path.stat().st_size,'sha256':None,'has_transparency':None,'error':''}
    try: row['sha256']=sha256_file(path)
    except Exception as exc: row['error']=f'hash failed: {type(exc).__name__}: {exc}'; return row
    try:
        with Image.open(path) as raw: raw.verify()
        with Image.open(path) as raw:
            raw.load(); oriented=ImageOps.exif_transpose(raw); w,h=oriented.size
            row.update(status='ok',format=raw.format,mode=oriented.mode,channels=len(oriented.getbands()),width=w,height=h,aspect_ratio=w/h if h else None,has_transparency=('A' in oriented.getbands() or 'transparency' in raw.info))
    except Exception as exc:
        status='unsupported_format' if path.suffix.lower() in {'.heic','.heif'} else ('corrupted' if isinstance(exc,OSError) else 'unreadable')
        row.update(status=status,error=f'{type(exc).__name__}: {exc}')
    return row
rows=[]
for idx,path in enumerate(tqdm(image_files,desc='Image audit')):
    try: rows.append(inspect_image(path))
    except Exception as exc: rows.append({'relative_path':path.relative_to(DATA_DIR).as_posix(),'filename':path.name,'extension':path.suffix.lower(),'status':'unreadable','format':None,'mode':None,'channels':None,'width':None,'height':None,'aspect_ratio':None,'file_bytes':path.stat().st_size,'sha256':None,'has_transparency':None,'error':f'{type(exc).__name__}: {exc}'})
    if (idx+1)%250==0: resource_log.append(resource_snapshot(f'image_audit_{idx+1}'))
inventory=pd.DataFrame(rows,columns=IMAGE_COLUMNS)
if inventory.empty: inventory=pd.DataFrame({c:pd.Series(dtype='string' if c in {'relative_path','filename','extension','status','format','mode','sha256','error'} else 'object') for c in IMAGE_COLUMNS})
inventory.to_csv(REPORT_DIR/'image_inventory.csv',index=False)
valid=inventory[inventory.status=='ok'].copy(); corrupt=inventory[inventory.status=='corrupted'].copy(); unsupported=inventory[inventory.status=='unsupported_format'].copy(); unreadable=inventory[inventory.status=='unreadable'].copy()
corrupt.to_csv(REPORT_DIR/'corrupted_images.csv',index=False); unsupported.to_csv(REPORT_DIR/'unsupported_images.csv',index=False); unreadable.to_csv(REPORT_DIR/'unreadable_images.csv',index=False)
if len(valid):
    dimensions={'readable_images':int(len(valid)),'width_min':int(valid.width.min()),'width_median':float(valid.width.median()),'width_max':int(valid.width.max()),'height_min':int(valid.height.min()),'height_median':float(valid.height.median()),'height_max':int(valid.height.max()),'aspect_ratio_median':float(valid.aspect_ratio.median()),'formats':valid.format.value_counts().to_dict(),'modes':valid.mode.value_counts().to_dict(),'transparent_count':int(valid.has_transparency.sum()),'file_bytes_min':int(valid.file_bytes.min()),'file_bytes_median':float(valid.file_bytes.median()),'file_bytes_max':int(valid.file_bytes.max())}
else: dimensions={'readable_images':0}
(REPORT_DIR/'dimension_summary.json').write_text(json.dumps(dimensions,indent=2))
if len(valid):
    counts=valid.groupby('sha256',dropna=True).size(); duplicate_hashes=counts[counts>1].index
    exact_dups=valid[valid.sha256.isin(duplicate_hashes)].sort_values(['sha256','relative_path']).copy()
else: exact_dups=pd.DataFrame(columns=IMAGE_COLUMNS)
exact_dups.to_csv(REPORT_DIR/'exact_duplicates.csv',index=False)
exact_groups=pd.DataFrame([{'sha256':key,'paths':' | '.join(g.relative_path.astype(str)),'count':len(g)} for key,g in exact_dups.groupby('sha256')],columns=['sha256','paths','count'])
exact_groups.to_csv(REPORT_DIR/'exact_duplicate_groups.csv',index=False)
print('Files:',len(all_files),'| images:',len(image_files),'| readable:',len(valid),'| exact duplicate groups:',len(exact_groups))

## Optional perceptual duplicates, annotation mapping, and verified labels

In [ ]:
ENABLE_PHASH=False
PHASH_THRESHOLD=6; PHASH_MAX_BUCKET=1000; PHASH_MAX_PAIRS=200000
phash_rows=[]; near_pairs=[]; phash_skipped=[]
if ENABLE_PHASH and len(valid):
    subprocess.run([sys.executable,'-m','pip','install','-q','ImageHash'],check=True); import imagehash
    for rel in tqdm(valid.relative_path.astype(str),desc='Perceptual hashes'):
        try:
            with Image.open(DATA_DIR/rel) as raw: phash_rows.append({'relative_path':rel,'phash':str(imagehash.phash(ImageOps.exif_transpose(raw).convert('RGB'),hash_size=8))})
        except Exception as exc: phash_skipped.append({'path':rel,'error':str(exc)})
pd.DataFrame(phash_rows,columns=['relative_path','phash']).to_csv(REPORT_DIR/'perceptual_hashes.csv',index=False)
pd.DataFrame(near_pairs,columns=['path_a','path_b','hamming_distance','threshold']).to_csv(REPORT_DIR/'perceptual_duplicate_candidates.csv',index=False)
(REPORT_DIR/'perceptual_hash_skipped.json').write_text(json.dumps(phash_skipped,indent=2))

annotation_manifest=[]; annotation_tables={}
for path in annotation_files:
    rel=path.relative_to(DATA_DIR).as_posix(); record={'relative_path':rel,'suffix':path.suffix.lower(),'status':'unreadable','row_count':None,'columns':[],'error':''}
    try:
        if path.suffix.lower()=='.csv': table=pd.read_csv(path); annotation_tables[rel]=table
        elif path.suffix.lower()=='.json':
            obj=json.loads(path.read_text(encoding='utf-8')); table=pd.DataFrame(obj) if isinstance(obj,list) and (not obj or isinstance(obj[0],dict)) else None
            if table is None: record.update(status='readable_non_tabular_json',row_count=len(obj) if isinstance(obj,list) else None); annotation_manifest.append(record); continue
            annotation_tables[rel]=table
        elif path.suffix.lower() in {'.xlsx','.xls'}: table=pd.read_excel(path); annotation_tables[rel]=table
        else: record.update(status='present_not_parsed'); annotation_manifest.append(record); continue
        record.update(status='readable',row_count=len(table),columns=[str(c) for c in table.columns])
    except Exception as exc: record['error']=f'{type(exc).__name__}: {exc}'
    annotation_manifest.append(record)
ann_manifest=pd.DataFrame(annotation_manifest,columns=['relative_path','suffix','status','row_count','columns','error'])
if ann_manifest.empty: ann_manifest=pd.DataFrame({'relative_path':pd.Series(dtype='string'),'suffix':pd.Series(dtype='string'),'status':pd.Series(dtype='string'),'row_count':pd.Series(dtype='object'),'columns':pd.Series(dtype='object'),'error':pd.Series(dtype='string')})
ann_manifest.to_csv(REPORT_DIR/'annotation_manifest.csv',index=False)
def norm_name(value): return re.sub(r'[^a-z0-9]+','_',str(value).lower()).strip('_')
def safe_key(value):
    text=str(value).strip().replace('\\','/')
    if text.startswith('/') or re.match(r'^[A-Za-z]:',text) or '..' in Path(text).parts: return None
    while text.startswith('./'): text=text[2:]
    return text.lower()
image_key_index={}
for rel in inventory.relative_path.astype(str):
    for key in {safe_key(rel),safe_key(Path(rel).name)}:
        if key: image_key_index.setdefault(key,[]).append(rel)
path_tokens=('file','filename','filepath','path','image','img','photo','picture'); label_tokens={'emotion','emotions','expression','mood','sentiment','species','animal','animal_type','animal_class','breed','label','class','category','target'}
map_rows=[]; label_candidates=[]; reference_rows=[]
for source,table in annotation_tables.items():
    path_cols=[]; label_cols=[]
    for col in table.columns:
        name=norm_name(col)
        if any(token in name for token in path_tokens): path_cols.append(col)
        if name in label_tokens or any(token in name for token in ('emotion','species','animal_type')): label_cols.append(col)
    for col in table.columns:
        if col not in path_cols:
            values=table[col].dropna().astype(str).head(100)
            if len(values) and sum(Path(v.replace('\\','/')).suffix.lower() in IMAGE_EXTS for v in values)/len(values)>=0.5: path_cols.append(col)
    for col in label_cols: label_candidates.append({'source':source,'column':str(col),'candidate_only_not_authoritative':True,'unique_values':int(table[col].nunique(dropna=True))})
    if not path_cols: map_rows.append({'source':source,'row_index':None,'annotation_reference':None,'status':'no_path_column','match_count':0,'matched_paths':'','candidate_labels_json':'{}'}); continue
    for row_index,row in table.iterrows():
        refs=[]; original=None
        for col in path_cols:
            if pd.notna(row[col]):
                original=str(row[col]); key=safe_key(original); basename=safe_key(Path(original.replace('\\','/')).name)
                refs.extend(image_key_index.get(key,[]) if key else []); refs.extend(image_key_index.get(basename,[]) if basename else [])
        matches=sorted(set(refs)); labels={str(col):str(row[col]) for col in label_cols if pd.notna(row[col])}
        map_rows.append({'source':source,'row_index':int(row_index),'annotation_reference':original,'status':'matched' if matches else 'unmatched_annotation','match_count':len(matches),'matched_paths':'|'.join(matches),'candidate_labels_json':json.dumps(labels,ensure_ascii=False)})
        for image_path in matches: reference_rows.append({'image_path':image_path,'source':source,'row_index':int(row_index),'candidate_labels_json':json.dumps(labels,ensure_ascii=False)})
mapping=pd.DataFrame(map_rows,columns=['source','row_index','annotation_reference','status','match_count','matched_paths','candidate_labels_json'])
if mapping.empty: mapping=pd.DataFrame(columns=['source','row_index','annotation_reference','status','match_count','matched_paths','candidate_labels_json'])
mapping.to_csv(REPORT_DIR/'annotation_image_mapping.csv',index=False); mapping[mapping.status=='unmatched_annotation'].to_csv(REPORT_DIR/'unmatched_annotations.csv',index=False)
refs=pd.DataFrame(reference_rows,columns=['image_path','source','row_index','candidate_labels_json'])
if refs.empty: refs=pd.DataFrame(columns=['image_path','source','row_index','candidate_labels_json'])
refs.to_csv(REPORT_DIR/'annotation_references.csv',index=False)
duplicate_refs=refs[refs.duplicated('image_path',keep=False)].sort_values('image_path') if len(refs) else refs.copy(); duplicate_refs.to_csv(REPORT_DIR/'duplicate_annotation_references.csv',index=False)
conflicts=[]
for image_path,group in refs.groupby('image_path') if len(refs) else []:
    values={}
    for payload in group.candidate_labels_json:
        for col,val in json.loads(payload).items(): values.setdefault(col,set()).add(val)
    for col,vals in values.items():
        if len(vals)>1: conflicts.append({'image_path':image_path,'column':col,'values':' | '.join(sorted(vals)),'reference_count':len(group)})
pd.DataFrame(conflicts,columns=['image_path','column','values','reference_count']).to_csv(REPORT_DIR/'conflicting_labels.csv',index=False)
matched_images=set(refs.image_path.astype(str)) if len(refs) else set(); unlabeled=inventory[~inventory.relative_path.astype(str).isin(matched_images)].copy() if len(inventory) else inventory.copy(); unlabeled.to_csv(REPORT_DIR/'unlabeled_images.csv',index=False)
pd.DataFrame(label_candidates,columns=['source','column','candidate_only_not_authoritative','unique_values']).to_csv(REPORT_DIR/'label_column_candidates.csv',index=False)
VERIFIED_LABEL_COLUMNS={}
verified=[]
for source,column in VERIFIED_LABEL_COLUMNS.items():
    if source not in annotation_tables or column not in annotation_tables[source].columns: continue
    table=annotation_tables[source]; selected=mapping[(mapping.source==source)&(mapping.match_count==1)&(mapping.status=='matched')]
    for _,record in selected.iterrows():
        label=table.iloc[int(record.row_index)][column]
        if pd.notna(label): verified.append({'source':source,'image_path':record.matched_paths,'label_column':column,'label':str(label)})
verified=pd.DataFrame(verified,columns=['source','image_path','label_column','label'])
class_dist=verified.groupby(['source','label_column','label']).size().reset_index(name='count') if len(verified) else pd.DataFrame(columns=['source','label_column','label','count'])
class_dist.to_csv(REPORT_DIR/'class_distribution.csv',index=False); verified.to_csv(REPORT_DIR/'verified_label_image_mapping.csv',index=False)
print('Unmatched annotations:',int((mapping.status=='unmatched_annotation').sum()) if len(mapping) else 0,'| unlabeled:',len(unlabeled),'| duplicate refs:',len(duplicate_refs),'| conflicts:',len(conflicts))

## Leakage assessment and Phase 1 verdict

In [ ]:
stem_counts=valid.assign(stem=valid.filename.map(lambda name:Path(name).stem)).groupby('stem').size() if len(valid) else pd.Series(dtype=int); repeated_stems=stem_counts[stem_counts>1]
subject_records=[]
for source,table in annotation_tables.items():
    for col in table.columns:
        if any(token in norm_name(col) for token in ('subject','individual','animal_id','patient','identity')):
            counts=table[col].dropna().astype(str).value_counts()
            for subject,count in counts[counts>1].items(): subject_records.append({'source':source,'column':str(col),'subject_identifier':subject,'rows':int(count),'interpretation':'possible overlap; identity semantics unverified'})
subject_df=pd.DataFrame(subject_records,columns=['source','column','subject_identifier','rows','interpretation']); subject_df.to_csv(REPORT_DIR/'possible_subject_overlap.csv',index=False)
leakage={'exact_duplicate_groups':int(len(exact_groups)),'files_in_exact_duplicate_groups':int(len(exact_dups)),'near_duplicate_candidates':int(len(near_pairs)),'near_duplicate_enabled':ENABLE_PHASH,'repeated_filename_stems':int(len(repeated_stems)),'possible_subject_overlap_records':len(subject_df),'unmatched_annotation_rows':int((mapping.status=='unmatched_annotation').sum()) if len(mapping) else 0,'unlabeled_images':len(unlabeled),'duplicate_annotation_references':len(duplicate_refs),'conflicting_label_records':len(conflicts),'split_created':False,'limitations':['Subject matching is heuristic','Perceptual search is capped','Repeated stems do not prove duplicates','No split or training performed']}
(REPORT_DIR/'leakage_assessment.json').write_text(json.dumps(leakage,indent=2))
issues=[]
for label,frame in [('corrupted',corrupt),('unsupported',unsupported),('unreadable',unreadable)]:
    if len(frame): issues.append(f'{len(frame)} {label} image candidates')
if len(exact_groups): issues.append(f'{len(exact_groups)} exact duplicate groups')
if not len(verified): issues.append('No analyst-verified label assignments configured')
suitability='potentially suitable after annotation, duplicate, and license review' if len(verified) and not len(corrupt) else 'not established for supervised training; review label evidence and audit findings first'
verdict={'phase':1,'dataset_suitability':suitability,'file_counts':{'all_files':len(all_files),'image_candidates':len(image_files),'readable':len(valid),'corrupted':len(corrupt),'unsupported':len(unsupported),'unreadable':len(unreadable),'annotation_candidates':len(annotation_files)},'detected_issues':issues,'known_limitations':leakage['limitations'],'unresolved_questions':['Are Kaggle terms/license suitable?','Is the annotation mapping authoritative?','Are species and emotion separate targets?'],'phase_2_recommendations':['Manually validate mappings','Review duplicate/conflicting/unlabeled records','Verify license and terms','Resolve leakage risks before splitting']}
(REPORT_DIR/'phase1_verdict.json').write_text(json.dumps(verdict,indent=2))
(REPORT_DIR/'audit_summary.json').write_text(json.dumps({'project':'AnimalEmotion AI','dataset_slug':DATASET_SLUG,'seed':SEED,'verdict':verdict,'leakage':leakage,'dimensions':dimensions,'verified_label_rows':len(verified)},indent=2))
resource_log.append(resource_snapshot('audit_complete')); pd.DataFrame(resource_log).to_csv(REPORT_DIR/'resource_monitoring.csv',index=False)
print(json.dumps(verdict,indent=2))

## Report validation and downloadable archive

In [ ]:
EXPECTED_REPORTS=['dataset_metadata_and_license_review.json','safe_extraction.json','folder_hierarchy.csv','image_inventory.csv','corrupted_images.csv','unsupported_images.csv','unreadable_images.csv','dimension_summary.json','exact_duplicates.csv','exact_duplicate_groups.csv','perceptual_hashes.csv','perceptual_duplicate_candidates.csv','perceptual_hash_skipped.json','annotation_manifest.csv','annotation_image_mapping.csv','unmatched_annotations.csv','annotation_references.csv','duplicate_annotation_references.csv','conflicting_labels.csv','unlabeled_images.csv','label_column_candidates.csv','class_distribution.csv','verified_label_image_mapping.csv','imbalance_analysis.csv','leakage_assessment.json','possible_subject_overlap.csv','phase1_verdict.json','audit_summary.json','resource_monitoring.csv']
missing=[name for name in EXPECTED_REPORTS if not (REPORT_DIR/name).is_file()]
if missing: raise RuntimeError('Expected reports missing: '+', '.join(missing))
manifest={'created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'report_directory':str(REPORT_DIR),'expected_reports':EXPECTED_REPORTS,'report_files':sorted(p.name for p in REPORT_DIR.iterdir() if p.is_file()),'archive_name':zip_path.name,'archive_sha256':archive_digest}
(REPORT_DIR/'REPORT_MANIFEST.json').write_text(json.dumps(manifest,indent=2))
bundle=PROJECT_DIR/'animalemotion_phase1_audit_reports.zip'
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(REPORT_DIR.iterdir()):
        if path.is_file(): archive.write(path,arcname='reports/'+path.name)
print('Reports validated:',len(EXPECTED_REPORTS),'| bundle:',bundle)
try:
    from google.colab import files as colab_files
    colab_files.download(str(bundle))
except Exception as exc: print('Browser download unavailable:',exc)
print('No training, splitting, or source-image modification.')

## Static notebook validation
This code-only check excludes itself and all Markdown text. It uses `nbformat.validate`, AST parsing, unique cell IDs, and AST-based detection of forbidden acquisition modules/calls.

In [ ]:
import nbformat
notebook_path=Path('/content/AnimalEmotion_AI_Phase1_Audit.ipynb')
if notebook_path.exists():
    nb=nbformat.read(notebook_path,as_version=4)
else:
    nb=globals().get('_NOTEBOOK_OBJECT',None)
if nb is None:
    print('Notebook JSON file is not available locally. Save/open this notebook in /content for full JSON validation.')
else:
    nbformat.validate(nb)
    ids=[c.get('id') for c in nb.cells]
    if len(ids)!=len(set(ids)) or any(not c for c in ids):
        raise AssertionError('Notebook cell IDs are missing or duplicated.')
    forbidden_imports={'kaggle','kagglehub'}
    forbidden_calls={'download','get'}
    forbidden_names={'KAGGLE_KEY','KAGGLE_USERNAME','KAGGLE_CONFIG_DIR'}
    violations=[]
    manual_upload=False
    for index,c in enumerate(nb.cells):
        if c.cell_type!='code' or c.get('id')=='static-validation':
            continue
        ast.parse(c.source,filename=f'cell_{index}')
        tree=ast.parse(c.source,filename=f'cell_{index}')
        for node in ast.walk(tree):
            if isinstance(node,ast.Import):
                for alias in node.names:
                    if alias.name.split('.')[0] in forbidden_imports:
                        violations.append(f'import {alias.name} in cell {index}')
            if isinstance(node,ast.ImportFrom):
                if node.module and node.module.split('.')[0] in forbidden_imports:
                    violations.append(f'import from {node.module} in cell {index}')
            if isinstance(node,ast.Name) and node.id in forbidden_names:
                violations.append(f'credential name {node.id} in cell {index}')
            if isinstance(node,ast.Call):
                func=node.func
                call_name=None
                if isinstance(func,ast.Attribute): call_name=func.attr
                elif isinstance(func,ast.Name): call_name=func.id
                if call_name=='upload': manual_upload=True
                if isinstance(func,ast.Attribute) and isinstance(func.value,ast.Attribute):
                    if getattr(func.value.attr,None)=='datasets': violations.append(f'Kaggle dataset API call in cell {index}')
                if isinstance(func,ast.Attribute) and isinstance(func.value,ast.Name):
                    if func.value.id=='kagglehub' or (func.value.id=='userdata' and func.attr=='get'):
                        violations.append(f'forbidden acquisition call {func.value.id}.{func.attr} in cell {index}')
                if isinstance(func,ast.Name) and func.id=='download':
                    violations.append(f'automatic download call in cell {index}')
    if not manual_upload:
        violations.append('manual files.upload() workflow not detected')
    if violations:
        raise AssertionError('Forbidden workflow or credential patterns detected: '+'; '.join(sorted(set(violations))))
    print('Notebook JSON valid; unique cell IDs valid; every executable code cell (except this validator) passed ast.parse; manual ZIP upload is the only acquisition workflow.')